# 09 Grad-CAM Explainability & Visualization

## Objective
Generate Grad-CAM heatmaps for model interpretability:
- ✓ Implement Grad-CAM algorithm
- ✓ Generate attention maps
- ✓ Visualize correct predictions
- ✓ Visualize misclassifications
- ✓ Compare attention patterns across classes
- ✓ Generate explainability report

**Prerequisites:** All previous notebooks (01-08)

### Import Libraries

import sys
import os

sys.path.insert(0, '../utils')

from model_utils import load_model

import numpy as np
import tensorflow as tf
import json
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print("✓ Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

### Load Data and Model

print("Loading test data and model...\n")

# Load test data
X_test = np.load('../results/preprocessed_data/X_test.npy')
y_test = np.load('../results/preprocessed_data/y_test.npy')

# Load label encoding
with open('../results/preprocessed_data/label_encoding.json', 'r') as f:
    label_encoding = json.load(f)

num_classes = len(label_encoding)
reverse_encoding = {v: k for k, v in label_encoding.items()}
class_names = [reverse_encoding[i] for i in range(num_classes)]

# Load best model
with open('../results/reports/best_model.json', 'r') as f:
    best_model_info = json.load(f)

best_model_name = best_model_info['name']

if 'mobilenet' in best_model_name.lower():
    model_path = '../models/mobilenet/mobilenet_v2.h5'
else:
    model_path = '../models/cnn/cnn_baseline.h5'

model = load_model(model_path)

print(f"✓ Model loaded: {best_model_name}")
print(f"✓ Test data shape: {X_test.shape}")

### 1. Implement Grad-CAM

class GradCAM:
    """Grad-CAM implementation for model interpretability"""
    
    def __init__(self, model, layer_name):
        self.model = model
        self.layer_name = layer_name
        self.grad_model = tf.keras.models.Model(
            [model.inputs],
            [model.get_layer(layer_name).output, model.output]
        )
    
    def generate(self, img_array, pred_index=None):
        """Generate Grad-CAM heatmap"""
        with tf.GradientTape() as tape:
            conv_outputs, predictions = self.grad_model(img_array)
            if pred_index is None:
                pred_index = tf.argmax(predictions[0])
            class_channel = predictions[:, pred_index]
        
        grads = tape.gradient(class_channel, conv_outputs)
        
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        
        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)
        
        heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
        return heatmap.numpy()
    
    def overlay_heatmap(self, img, heatmap, alpha=0.4):
        """Overlay heatmap on original image"""
        heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
        
        heatmap_color = cm.jet(heatmap_resized)
        heatmap_rgb = (heatmap_color[:, :, :3] * 255).astype(np.uint8)
        
        overlaid = cv2.addWeighted(img.astype(np.uint8), 1 - alpha, heatmap_rgb, alpha, 0)
        return overlaid


# Find last convolutional layer
last_conv_layer = None
for layer in model.layers[::-1]:
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer.name
        break

print(f"✓ Grad-CAM initialized")
print(f"  Using layer: {last_conv_layer}")

grad_cam = GradCAM(model, last_conv_layer)

### 2. Generate Predictions for Analysis

print("\nGenerating predictions for Grad-CAM analysis...\n")

predictions = model.predict(X_test, verbose=0)
pred_labels_idx = np.argmax(predictions, axis=1)
true_labels_idx = np.argmax(y_test, axis=1)
confidence = np.max(predictions, axis=1)

# Identify correct and incorrect
correct_mask = (pred_labels_idx == true_labels_idx)
incorrect_mask = ~correct_mask

correct_indices = np.where(correct_mask)[0]
incorrect_indices = np.where(incorrect_mask)[0]

print(f"Correct predictions: {len(correct_indices)}")
print(f"Incorrect predictions: {len(incorrect_indices)}")

### 3. Grad-CAM for Correct Predictions

print("\nGenerating Grad-CAM for correct predictions...\n")

# Sample 6 correct predictions
sample_correct = np.random.choice(correct_indices, min(6, len(correct_indices)), replace=False)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for idx, img_idx in enumerate(sample_correct):
    # Original image
    img_array = X_test[img_idx:img_idx+1]
    img = (X_test[img_idx] * 255).astype(np.uint8)
    
    # Generate Grad-CAM
    heatmap = grad_cam.generate(img_array)
    
    # Overlay
    overlaid = grad_cam.overlay_heatmap(img, heatmap, alpha=0.5)
    
    # Plot original
    axes[0, idx].imshow(img)
    axes[0, idx].set_title(f"{class_names[true_labels_idx[img_idx]]}\nConf: {confidence[img_idx]:.3f}",
                           fontweight='bold', fontsize=9)
    axes[0, idx].axis('off')
    
    # Plot Grad-CAM
    axes[1, idx].imshow(overlaid)
    axes[1, idx].axis('off')

plt.suptitle('Grad-CAM: Correct Predictions', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('../results/plots/gradcam_correct_predictions.png', dpi=300, bbox_inches='tight')
print("✓ Saved: gradcam_correct_predictions.png")

### 4. Grad-CAM for Misclassifications

if len(incorrect_indices) > 0:
    print("\nGenerating Grad-CAM for misclassifications...\n")
    
    # Sample 6 incorrect predictions
    sample_incorrect = np.random.choice(incorrect_indices, min(6, len(incorrect_indices)), replace=False)
    
    fig, axes = plt.subplots(3, 6, figsize=(18, 9))
    
    for idx, img_idx in enumerate(sample_incorrect):
        # Original image
        img_array = X_test[img_idx:img_idx+1]
        img = (X_test[img_idx] * 255).astype(np.uint8)
        
        # Generate Grad-CAM for true class
        heatmap_true = grad_cam.generate(img_array, pred_index=true_labels_idx[img_idx])
        
        # Generate Grad-CAM for predicted class
        heatmap_pred = grad_cam.generate(img_array, pred_index=pred_labels_idx[img_idx])
        
        # Overlay
        overlaid_true = grad_cam.overlay_heatmap(img, heatmap_true, alpha=0.5)
        overlaid_pred = grad_cam.overlay_heatmap(img, heatmap_pred, alpha=0.5)
        
        # Plot original
        axes[0, idx].imshow(img)
        axes[0, idx].set_title(f"Original", fontweight='bold', fontsize=9)
        axes[0, idx].axis('off')
        
        # Plot true class Grad-CAM
        axes[1, idx].imshow(overlaid_true)
        axes[1, idx].set_title(f"True: {class_names[true_labels_idx[img_idx]]}",
                               fontweight='bold', fontsize=9, color='green')
        axes[1, idx].axis('off')
        
        # Plot predicted class Grad-CAM
        axes[2, idx].imshow(overlaid_pred)
        axes[2, idx].set_title(f"Pred: {class_names[pred_labels_idx[img_idx]]}",
                               fontweight='bold', fontsize=9, color='red')
        axes[2, idx].axis('off')
    
    plt.suptitle('Grad-CAM: Misclassifications (True vs Predicted Class)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/plots/gradcam_misclassifications.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: gradcam_misclassifications.png")
else:
    print("No misclassifications found!")

### 5. Grad-CAM by Class

print("\nGenerating Grad-CAM for each class...\n")

for class_idx, class_name in enumerate(class_names[:5]):  # Show first 5 classes
    # Find correct predictions for this class
    class_correct_mask = (true_labels_idx == class_idx) & correct_mask
    class_indices = np.where(class_correct_mask)[0]
    
    if len(class_indices) > 0:
        # Sample up to 4 images
        sample_class_idx = np.random.choice(class_indices, min(4, len(class_indices)), replace=False)
        
        fig, axes = plt.subplots(2, 4, figsize=(14, 6))
        
        for idx, img_idx in enumerate(sample_class_idx):
            # Original image
            img_array = X_test[img_idx:img_idx+1]
            img = (X_test[img_idx] * 255).astype(np.uint8)
            
            # Generate Grad-CAM
            heatmap = grad_cam.generate(img_array, pred_index=class_idx)
            
            # Overlay
            overlaid = grad_cam.overlay_heatmap(img, heatmap, alpha=0.5)
            
            # Plot original
            axes[0, idx].imshow(img)
            axes[0, idx].set_title(f"Conf: {confidence[img_idx]:.3f}", fontweight='bold', fontsize=9)
            axes[0, idx].axis('off')
            
            # Plot Grad-CAM
            axes[1, idx].imshow(overlaid)
            axes[1, idx].axis('off')
        
        # Hide unused subplots
        for idx in range(len(sample_class_idx), 4):
            axes[0, idx].axis('off')
            axes[1, idx].axis('off')
        
        plt.suptitle(f'Grad-CAM: Class "{class_name}"', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'../results/plots/gradcam_class_{class_idx}_{class_name.replace(" ", "_").lower()}.png',
                    dpi=300, bbox_inches='tight')
        print(f"✓ Saved: gradcam_class_{class_idx}_{class_name.lower().replace(' ', '_')}.png")

### 6. Heatmap Analysis

print("\n" + "="*70)
print("HEATMAP ANALYSIS")
print("="*70 + "\n")

# Analyze heatmap properties for correct predictions
heatmap_stats = []

for img_idx in correct_indices[:100]:  # Analyze first 100
    img_array = X_test[img_idx:img_idx+1]
    heatmap = grad_cam.generate(img_array)
    
    heatmap_stats.append({
        'mean_activation': heatmap.mean(),
        'max_activation': heatmap.max(),
        'std_activation': heatmap.std(),
        'concentration': (heatmap > heatmap.mean()).sum() / heatmap.size
    })

print("Heatmap Statistics (Correct Predictions):")
print(f"  Mean Activation: {np.mean([s['mean_activation'] for s in heatmap_stats]):.4f}")
print(f"  Max Activation: {np.mean([s['max_activation'] for s in heatmap_stats]):.4f}")
print(f"  Std Activation: {np.mean([s['std_activation'] for s in heatmap_stats]):.4f}")
print(f"  Avg Concentration: {np.mean([s['concentration'] for s in heatmap_stats]):.4f}")

### 7. Generate Explainability Report

print("\nGenerating explainability report...\n")

explainability_report = {
    'model': best_model_name,
    'method': 'Grad-CAM',
    'interpretation_layer': last_conv_layer,
    'summary': {
        'correct_predictions': int(len(correct_indices)),
        'incorrect_predictions': int(len(incorrect_indices)),
        'total_predictions': int(len(X_test)),
        'accuracy': float(len(correct_indices) / len(X_test))
    },
    'heatmap_analysis': {
        'mean_activation': float(np.mean([s['mean_activation'] for s in heatmap_stats])),
        'max_activation': float(np.mean([s['max_activation'] for s in heatmap_stats])),
        'std_activation': float(np.mean([s['std_activation'] for s in heatmap_stats])),
        'avg_concentration': float(np.mean([s['concentration'] for s in heatmap_stats]))
    },
    'insights': [
        'Grad-CAM reveals which image regions contribute to model predictions',
        'High concentration values indicate focused attention on specific features',
        'Comparing true class vs predicted class heatmaps shows misclassification causes',
        'Heatmaps should concentrate on disease/health indicators for reliability'
    ]
}

with open('../results/reports/explainability_report.json', 'w') as f:
    json.dump(explainability_report, f, indent=4)

print("✓ Explainability report saved")
print(json.dumps(explainability_report, indent=2))

### 8. Create Interpretation Guide

interpretation_guide = """
# Grad-CAM Interpretation Guide

## What is Grad-CAM?
Gradient-weighted Class Activation Mapping (Grad-CAM) produces visual explanations 
for decisions from a large class of Convolutional Neural Networks (CNNs).

## How to Read the Visualizations:

1. **Heatmap Colors**:
   - Red/Warm colors: High model attention (important regions)
   - Blue/Cool colors: Low model attention (less important regions)
   - Green: Medium attention

2. **Correct Predictions**:
   - Should show heatmaps concentrated on disease/health indicators
   - Red regions should align with visual symptoms
   - Indicates reliable decision-making

3. **Misclassifications**:
   - Comparing true vs predicted class heatmaps reveals why model made the error
   - If heatmaps overlap, model confused similar features
   - If heatmaps differ greatly, model attended to wrong regions

## Trust Indicators:

✓ **High Trust**: 
  - Focused heatmaps on disease/symptom regions
  - Heatmap concentrates on relevant areas
  - Consistent across same disease class

⚠️ **Medium Trust**:
  - Heatmaps somewhat diffuse
  - Some attention on irrelevant regions
  - Works but could be improved

✗ **Low Trust**:
  - Uniform heatmaps across entire image
  - Random attention patterns
  - Model likely making random decisions

## Generated Visualizations:

1. **gradcam_correct_predictions.png**: Correctly classified images
2. **gradcam_misclassifications.png**: Misclassified images with true vs predicted class attention
3. **gradcam_class_*.png**: Attention patterns per disease/health class

## Recommendations:

- Use Grad-CAM to validate model reliability before deployment
- If heatmaps look suspicious, gather more training data
- Fine-tune model if attention patterns are unfocused
- Combine with domain expert review for production deployment
"""

with open('../results/reports/gradcam_interpretation_guide.txt', 'w') as f:
    f.write(interpretation_guide)

print("✓ Interpretation guide saved")

### 9. Summary

print("\n" + "="*70)
print("✓ GRAD-CAM ANALYSIS COMPLETE")
print("="*70)

print(f"\n📊 Analysis Summary:")
print(f"  Model: {best_model_name}")
print(f"  Interpretation Layer: {last_conv_layer}")
print(f"  Correct Predictions Analyzed: {len(correct_indices)}")
print(f"  Incorrect Predictions Analyzed: {len(incorrect_indices)}")

print(f"\n📈 Heatmap Characteristics:")
print(f"  Mean Activation: {np.mean([s['mean_activation'] for s in heatmap_stats]):.4f}")
print(f"  Max Activation: {np.mean([s['max_activation'] for s in heatmap_stats]):.4f}")
print(f"  Avg Concentration: {np.mean([s['concentration'] for s in heatmap_stats]):.4f}")

print(f"\n💾 Files Generated:")
print(f"  - gradcam_correct_predictions.png")
print(f"  - gradcam_misclassifications.png")
print(f"  - gradcam_class_*.png (per-class analysis)")
print(f"  - explainability_report.json")
print(f"  - gradcam_interpretation_guide.txt")

print(f"\n✓ All notebooks completed successfully!")
print(f"  Review explainability visualizations in: ../results/plots/")
print(f"  Read interpretation guide: ../results/reports/gradcam_interpretation_guide.txt")